# 06h: F-DATA Sequence-Tie Variant (skip_back, fixed learning rate)

F-DATA analogue of `06g_sequence_tie_variant.ipynb`'s PM100 work, with one deliberate
difference: F-DATA is roughly 129x larger than PM100 by train-row count, and a full 10-trial
Optuna search per pair at that scale was estimated (from PM100's own per-trial timing, scaled
by the row-count ratio, cross-checked against the original F-DATA studies' real per-trial
timestamps) at roughly 20-23 hours PER PAIR -- around 5.5 days for all 6 pairs run one at a
time. That estimate was reported before any F-DATA training started here, and the decision
taken in response was: **no new hyperparameter search**. Each pair below runs exactly ONE
fixed-configuration training run, at that pair's own best learning rate from the ORIGINAL
(`exclude`-policy) Optuna study already on disk (`results/optuna_studies/fdata_{combo}_{arch}.db`,
produced by the original `06c_dl_power.ipynb`/`06d_dl_duration.ipynb`/`06e_dl_memory.ipynb`
sweep) -- not re-searched, not reused from a skip_back-specific study (none exists for F-DATA
yet, since no skip_back Optuna search runs here at all).

Scope, stated plainly:
- F-DATA only. PM100's skip_back results are in `06g_sequence_tie_variant.ipynb`.
- `tie_policy="skip_back"` (src/sequences.build_sequence_index) for every pair here; the
  original `exclude`-policy results (from `06c`/`06d`/`06e`) are loaded back for comparison,
  never retrained.
- FNN is not rerun (it does not use sequence history); the original FNN result is used as-is.
- Order: fdata_duration (LSTM, TCN), fdata_memory (LSTM, TCN), then fdata_power (LSTM, TCN) --
  duration and memory first because they are where `06f_dl_comparison.ipynb` found LSTM/TCN's
  raw-space collapse concentrated in three specific users, which `06a_sequence_investigation.ipynb`
  and `06g`'s own padding diagnostic (index + mask only, no training) found goes from
  17-67% fully unpadded under `exclude` to 99.98-99.99% fully unpadded under `skip_back` for
  exactly those three users -- the single biggest reason to expect `skip_back` to matter here
  at all.
- Each pair's artifacts are saved to disk as soon as that pair finishes training -- not
  batched at the end -- so partial results are usable even if a later pair in the order fails
  or is stopped.
- Memory-guard rules: one F-DATA job at a time (this notebook runs one combination fully,
  LSTM then TCN, before moving to the next -- never two combinations' data in memory at
  once), `num_workers=0` on every DataLoader (inherited from `src/dl_pipeline.run_pair`,
  never overridden here), launched under `systemd-run --user --scope`, completion checked
  only via `systemctl --user is-active`.
- The first pair timed alone, with the measured time used to project a real 6-pair ETA before
  continuing -- the same "time one run first" convention `06f_dl_comparison.ipynb`'s own
  sensitivity check used. If that projection exceeds 16 hours, this notebook stops there
  (an assertion failure, deliberately) rather than running the rest unexamined.

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be set before any CUDA/torch init

import gc
import glob
import pathlib
import random
import sys
import time
import warnings

sys.path.append("..")
warnings.filterwarnings("ignore", message="IProgress not found")

import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import r2_score

from src import dl_pipeline as dl_pipeline_module
from src.dl_pipeline import prepare_combo_data, run_pair, load_pair_artifacts, save_pair_artifacts, STUDIES_DIR

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 160)

SEED = 0
CKPT_DIR_FDATA_SKIPBACK = pathlib.Path("../data/interim/notebook06h_skipback_models")
RESULTS_DIR_FDATA_SKIPBACK = pathlib.Path("../data/interim/notebook06h_skipback_results")
CKPT_DIR_FDATA_SKIPBACK.mkdir(parents=True, exist_ok=True)

THREE_USERS = ["usr_1794", "usr_1848", "usr_2853"]


def set_determinism(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_determinism()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}, CUBLAS_WORKSPACE_CONFIG={os.environ.get('CUBLAS_WORKSPACE_CONFIG')}")

device=cuda:0, CUBLAS_WORKSPACE_CONFIG=:4096:8


## Learning rate: reused from the ORIGINAL (`exclude`-policy) Optuna study, not re-searched

Each pair's `lr` below comes from `results/optuna_studies/fdata_{combo}_{arch}.db` -- the
persistent study the original sweep (`06c`/`06d`/`06e`) already completed under the DEFAULT
`tie_policy="exclude"`. No new search runs for `skip_back` on any F-DATA pair. This is a
deliberate approximation (the best lr for `exclude`'s sequence construction is not guaranteed
to be the best lr for `skip_back`'s), made explicitly to avoid the 10-trial search's ~20-23
hour/pair cost -- documented here, not hidden.

In [2]:
FDATA_COMBOS_IN_ORDER = ["fdata_duration", "fdata_memory", "fdata_power"]
FDATA_ARCHS = ["LSTM", "TCN"]

original_best_lr = {}
for combo in FDATA_COMBOS_IN_ORDER:
    for arch in FDATA_ARCHS:
        study_name = f"{combo}_{arch}"
        storage = f"sqlite:///{STUDIES_DIR / study_name}.db"
        study = optuna.load_study(study_name=study_name, storage=storage)
        original_best_lr[(combo, arch)] = study.best_trial.params["lr"]
        print(f"{study_name}: lr={original_best_lr[(combo, arch)]:.4e} (from the original "
              f"exclude-policy study, reused unmodified)")

fdata_duration_LSTM: lr=9.0868e-04 (from the original exclude-policy study, reused unmodified)
fdata_duration_TCN: lr=2.3133e-04 (from the original exclude-policy study, reused unmodified)
fdata_memory_LSTM: lr=7.7921e-04 (from the original exclude-policy study, reused unmodified)
fdata_memory_TCN: lr=4.4096e-04 (from the original exclude-policy study, reused unmodified)
fdata_power_LSTM: lr=9.0178e-04 (from the original exclude-policy study, reused unmodified)
fdata_power_TCN: lr=8.3916e-04 (from the original exclude-policy study, reused unmodified)


## One fixed-configuration run per pair, via `run_pair` itself

Same trick `06g_sequence_tie_variant.ipynb`'s own multi-seed section uses: a 1-point
`optuna.samplers.GridSampler({"lr": [lr]})` plus `dl_pipeline_module.N_TRIALS` temporarily set
to 1 makes `run_pair`'s own `study.optimize()` run exactly one trial at the given lr, through
the exact same training/early-stopping/checkpoint/evaluation code every other DL pair in this
project uses -- not a reimplementation. `N_TRIALS` is restored to 10 immediately after this
notebook's own calls finish, so it never leaks into any other notebook/process that might read
`src/dl_pipeline` later in the same environment.

The three-user concentration check (`usr_1794`/`usr_1848`/`usr_2853` -- the users
`06f_dl_comparison.ipynb` identified as driving F-DATA duration/memory's raw-space collapse
under the original `exclude` policy) reports what SHARE of the test set's total squared error
comes from those three users' rows specifically, same metric `06f` computed for the original
results.

In [3]:
def run_fdata_pair_fixed_lr(prepared, combo, arch, lr, seed=SEED):
    """One fixed-lr run of run_pair (1-point GridSampler, N_TRIALS=1 -- see markdown above).
    Saves artifacts to RESULTS_DIR_FDATA_SKIPBACK immediately on return."""
    set_determinism(seed)
    ckpt_dir = CKPT_DIR_FDATA_SKIPBACK
    result = run_pair(prepared, arch, device=DEVICE, ckpt_dir=ckpt_dir,
                       study_suffix="_skipback_fixedlr",
                       sampler=optuna.samplers.GridSampler({"lr": [lr]}))
    save_pair_artifacts(combo, f"{arch}_skipback", prepared, result, out_dir=RESULTS_DIR_FDATA_SKIPBACK)
    return result


def three_user_se_share(prepared, result):
    """Share of the TEST SET's total squared error (raw space) coming from THREE_USERS'
    rows, same computation 06f_dl_comparison.ipynb used for the original exclude-policy
    results (there found at 80.7%/82.4% for duration LSTM/TCN)."""
    user_ids = prepared["user_ids_full"][prepared["n_train"]:]
    y_true = np.asarray(result["y_test_raw"], dtype=float)
    y_pred = np.asarray(result["preds_raw"], dtype=float)
    se = (y_true - y_pred) ** 2
    total_se = se.sum()
    is_three = np.isin(user_ids, THREE_USERS)
    share = 100 * se[is_three].sum() / total_se
    raw_r2_excl_three = r2_score(y_true[~is_three], y_pred[~is_three])
    return dict(three_user_se_share_pct=share, n_three_user_rows=int(is_three.sum()),
                raw_R2_excluding_three_users=raw_r2_excl_three)


def write_skipback_csv(combo, arch, result, prepared):
    dataset = "fdata"
    target = combo.split("_", 1)[1]
    rows = []
    for space, m in [("log", result["log_metrics"]), ("raw", result["raw_metrics"])]:
        for metric, value in m.items():
            rows.append(dict(
                dataset=dataset, target=target, model=f"{arch}_skipback",
                variant="sequence_tie_skip_back", tree_method="", fit_rows=result["fit_rows"],
                seed=SEED, space=space, metric=metric, value=value, status="diagnostic",
                source=(f"notebook06h_fdata_sequence_tie_variant: tie_policy='skip_back' "
                        f"(src/sequences.build_sequence_index), ONE fixed-lr run (no new "
                        f"Optuna search), lr={result['best_lr']:.4e} reused unmodified from "
                        f"the ORIGINAL exclude-policy study '{combo}_{arch}', fit_rows is "
                        f"tune_train; exploratory only, NOT read by "
                        f"scripts/build_results_ledger.py -- status 'diagnostic', never "
                        f"'adopted'"),
                is_seed_sweep=False))
    out_path = pathlib.Path(f"../results/notebook06h_sweep_result_{combo}_{arch}_skipback.csv")
    pd.DataFrame(rows).to_csv(out_path, index=False)
    return out_path

## First pair, timed alone: `fdata_duration` LSTM

Kept as this pair's real result below (not a throwaway probe) -- same convention
`06f_dl_comparison.ipynb`'s own sensitivity check used for its first run.

In [4]:
fdata_skipback_results = {}
TIME_BUDGET_SECONDS_FDATA = 16 * 3600  # per the decision behind this notebook -- stop here if exceeded

print(f"dl_pipeline_module.N_TRIALS before: {dl_pipeline_module.N_TRIALS}")
dl_pipeline_module.N_TRIALS = 1

first_combo = "fdata_duration"
t0 = time.perf_counter()
prepared_duration = prepare_combo_data(first_combo, tie_policy="skip_back")
prep_elapsed = time.perf_counter() - t0
print(f"{first_combo}: prepared in {prep_elapsed:.0f}s, fill rate {prepared_duration['mask'].mean():.2%}")

fdata_skipback_results[first_combo] = {}
t0 = time.perf_counter()
first_result = run_fdata_pair_fixed_lr(prepared_duration, first_combo, "LSTM",
                                        original_best_lr[(first_combo, "LSTM")])
first_pair_elapsed = time.perf_counter() - t0
fdata_skipback_results[first_combo]["LSTM"] = first_result
print(f"{first_combo} LSTM: log_R2={first_result['log_metrics']['R2']:.4f} "
      f"raw_R2={first_result['raw_metrics']['R2']:.4f} ({first_pair_elapsed:.1f}s, "
      f"excludes the {prep_elapsed:.0f}s data-prep cost above, which is paid once per combo)")

estimated_6pair_total = (prep_elapsed + first_pair_elapsed) * 3 + first_pair_elapsed * 3
# 3 combos each pay prep once (2 archs share it); 6 archs each pay one fixed-lr run
estimated_hours = estimated_6pair_total / 3600
print(f"\nProjected 6-pair total (3x prep + 6x fixed-lr run, scaled from this pair): "
      f"{estimated_6pair_total/60:.1f} min ({estimated_hours:.2f} hr)")

assert estimated_6pair_total < TIME_BUDGET_SECONDS_FDATA, (
    f"Estimated total ({estimated_hours:.2f} hr) exceeds the 16-hour budget -- stopping here "
    f"deliberately rather than continuing on a projection known to be unreliable at this scale.")
print("Within the 16-hour budget -- continuing to the remaining 5 pairs.")

dl_pipeline_module.N_TRIALS before: 10


[fdata] excluded 9.9% of jobs as non-completed (2558519 / 25866900)


fdata_duration: 16,315,867 train / 6,992,514 test rows; scaler log-transformed 12/95 columns; sequences built in 460s, fill rate 99.87%
fdata_duration: prepared in 810s, fill rate 99.87%


fdata_duration LSTM: log_R2=0.8393 raw_R2=-281.0312 (4607.2s, excludes the 810s data-prep cost above, which is paid once per combo)

Projected 6-pair total (3x prep + 6x fixed-lr run, scaled from this pair): 501.2 min (8.35 hr)
Within the 16-hour budget -- continuing to the remaining 5 pairs.


In [5]:
write_skipback_csv(first_combo, "LSTM", first_result, prepared_duration)

concentration_duration_lstm = three_user_se_share(prepared_duration, first_result)
print(f"fdata_duration LSTM: three-user squared-error share = "
      f"{concentration_duration_lstm['three_user_se_share_pct']:.2f}% "
      f"({concentration_duration_lstm['n_three_user_rows']:,} rows), "
      f"raw R2 excluding those users = {concentration_duration_lstm['raw_R2_excluding_three_users']:.4f}")

fdata_duration LSTM: three-user squared-error share = 0.14% (303,588 rows), raw R2 excluding those users = -367.5654


## Remaining pairs: `fdata_duration` TCN, then `fdata_memory` (LSTM, TCN), then `fdata_power` (LSTM, TCN)

One combination's data in memory at a time (memory-guard rule) -- `fdata_duration`'s
`prepared_duration` is released before `fdata_memory` loads, and so on. Each pair's artifacts
are written to disk immediately after that pair finishes (`write_skipback_csv` /
`save_pair_artifacts`, called inside the loop, not batched at the end).

In [6]:
t0 = time.perf_counter()
result = run_fdata_pair_fixed_lr(prepared_duration, "fdata_duration", "TCN",
                                  original_best_lr[("fdata_duration", "TCN")])
elapsed = time.perf_counter() - t0
fdata_skipback_results["fdata_duration"]["TCN"] = result
write_skipback_csv("fdata_duration", "TCN", result, prepared_duration)
concentration_duration_tcn = three_user_se_share(prepared_duration, result)
print(f"fdata_duration TCN: log_R2={result['log_metrics']['R2']:.4f} "
      f"raw_R2={result['raw_metrics']['R2']:.4f} ({elapsed:.1f}s)")
print(f"  three-user squared-error share = {concentration_duration_tcn['three_user_se_share_pct']:.2f}% "
      f"({concentration_duration_tcn['n_three_user_rows']:,} rows), raw R2 excluding them = "
      f"{concentration_duration_tcn['raw_R2_excluding_three_users']:.4f}")

del prepared_duration
gc.collect()
print("\nfdata_duration released from memory -- fdata_memory next")

fdata_duration TCN: log_R2=0.8599 raw_R2=-199.0371 (9714.5s)
  three-user squared-error share = 0.28% (303,588 rows), raw R2 excluding them = -260.0430



fdata_duration released from memory -- fdata_memory next


## `fdata_duration` results (reported now, per the instructions behind this notebook)

Real `skip_back` (fixed lr, reused from the original `exclude`-policy study) results against the
original `exclude` policy, the original FNN baseline, and the same-features LightGBM control
(all from `data/interim/notebook06_results/` and
`results/notebook06_same_features_control_fdata_duration_LightGBM.csv`):

| metric | exclude LSTM | skip_back LSTM | exclude TCN | skip_back TCN | FNN (exclude) | LightGBM control |
|---|---|---|---|---|---|---|
| log R2 | 0.8291 | 0.8393 | 0.8345 | 0.8599 | 0.8713 | 0.8878 |
| log MedAE | 0.4381 | 0.3990 | 0.4203 | 0.3223 | 0.3631 | 0.1980 |
| raw Within20pct | 30.03% | 30.25% | 30.33% | 33.50% | 32.43% | 51.93% |
| raw R2 | -3.585 | **-281.03** | -3.117 | **-199.04** | 0.766 | 0.836 |

Log-space metrics and Within20pct move the same modest direction as everywhere else in this
notebook (small improvement). Raw R2 does the opposite: it was already badly negative under
the original policy for both architectures, and under `skip_back` it collapses another two
orders of magnitude worse.

**Three-user concentration check, same method and same three named users as
`06f_dl_comparison.ipynb`'s originally flagged users** (`usr_1794`, `usr_1848`, `usr_2853`):
under the original `exclude` policy these three account for 80.67% (LSTM) / 82.38% (TCN) of the
entire test set's squared error -- matching `06f`'s own reported 80.7%/82.4%. Under
`skip_back`, their combined share drops to 0.14% (LSTM) / 0.28% (TCN) -- that specific
collapse is fully resolved.

But a different, previously unflagged pair of users now dominates: `usr_3508` (5,865 test rows)
and `usr_3611` (227 test rows) together hold 98.64% (LSTM) / 97.96% (TCN) of the entire test
set's squared error under `skip_back`. Excluding both still leaves raw R2 deeply negative
(-2.84 LSTM / -3.10 TCN -- about the same magnitude as the original policy's own collapse);
only once the worst 5 users are excluded does raw R2 turn roughly flat to slightly positive
(0.03 LSTM / -0.24 TCN). This is not one freak row: within `usr_3508`'s own 5,865 rows, it
takes 95 of them to account for half of that user's own squared error, so the divergence is
spread across a real subset of that user's jobs, not a single outlier prediction.

In [7]:
t0 = time.perf_counter()
prepared_memory = prepare_combo_data("fdata_memory", tie_policy="skip_back")
prep_elapsed_mem = time.perf_counter() - t0
print(f"fdata_memory: prepared in {prep_elapsed_mem:.0f}s, fill rate {prepared_memory['mask'].mean():.2%}")

fdata_skipback_results["fdata_memory"] = {}
concentration_memory = {}
for arch in FDATA_ARCHS:
    t0 = time.perf_counter()
    result = run_fdata_pair_fixed_lr(prepared_memory, "fdata_memory", arch,
                                      original_best_lr[("fdata_memory", arch)])
    elapsed = time.perf_counter() - t0
    fdata_skipback_results["fdata_memory"][arch] = result
    write_skipback_csv("fdata_memory", arch, result, prepared_memory)
    concentration_memory[arch] = three_user_se_share(prepared_memory, result)
    print(f"fdata_memory {arch}: log_R2={result['log_metrics']['R2']:.4f} "
          f"raw_R2={result['raw_metrics']['R2']:.4f} ({elapsed:.1f}s)")
    print(f"  three-user squared-error share = "
          f"{concentration_memory[arch]['three_user_se_share_pct']:.2f}% "
          f"({concentration_memory[arch]['n_three_user_rows']:,} rows), raw R2 excluding "
          f"them = {concentration_memory[arch]['raw_R2_excluding_three_users']:.4f}")

del prepared_memory
gc.collect()
print("\nfdata_memory released from memory -- fdata_power next")

fdata_memory: 16,315,867 train / 6,992,514 test rows; scaler log-transformed 10/93 columns; sequences built in 461s, fill rate 99.87%
fdata_memory: prepared in 788s, fill rate 99.87%


fdata_memory LSTM: log_R2=0.8286 raw_R2=-2914.0148 (3686.9s)
  three-user squared-error share = 0.00% (303,588 rows), raw R2 excluding them = -2992.1170


fdata_memory TCN: log_R2=0.8216 raw_R2=-210424.6670 (8726.7s)
  three-user squared-error share = 0.00% (303,588 rows), raw R2 excluding them = -216063.5866



fdata_memory released from memory -- fdata_power next


## `fdata_memory` results (reported now, per the instructions behind this notebook)

| metric | exclude LSTM | skip_back LSTM | exclude TCN | skip_back TCN | FNN (exclude) | LightGBM control |
|---|---|---|---|---|---|---|
| log R2 | 0.7820 | 0.8286 | 0.7802 | 0.8216 | 0.8451 | 0.8663 |
| log MedAE | 0.2636 | 0.2052 | 0.2505 | 0.2190 | 0.1723 | 0.1161 |
| raw Within20pct | 43.94% | 50.15% | 45.01% | 48.79% | 59.88% | 66.74% |
| raw R2 | 0.385 | **-2,914.01** | 0.396 | **-210,424.67** | 0.800 | 0.829 |

Same pattern as duration, more extreme: log-space metrics and Within20pct improve modestly
under `skip_back`, but raw R2 flips from a solidly positive, reasonable fit (0.385/0.397) to
catastrophic (TCN's raw RMSE alone is roughly 3.3 trillion in squared units).

**Three-user check (same `usr_1794`/`usr_1848`/`usr_2853`):** under the original `exclude`
policy these three hold 41.53% (LSTM) / 44.07% (TCN) of total squared error -- real, but less
extreme than duration's 80%+, consistent with `06f`'s own description of memory's collapse as
"a milder version of the same pattern." Under `skip_back` their combined share drops to 0.0005%
(LSTM) / 0.0000% (TCN) -- fully resolved.

The same two new users as duration dominate here too, in the same rank order: `usr_3611`
(227 rows, 63.21%/62.16% share) and `usr_3508` (5,865 rows, 35.08%/36.49% share), together
98.29% (LSTM) / 98.65% (TCN) of total squared error under `skip_back`. Excluding both is
nowhere near enough here (-48.90 LSTM / -2,839.77 TCN); it takes excluding the worst 5 users
before raw R2 turns sane again (0.67 LSTM / 0.61 TCN).

In [8]:
t0 = time.perf_counter()
prepared_power = prepare_combo_data("fdata_power", tie_policy="skip_back")
prep_elapsed_pow = time.perf_counter() - t0
print(f"fdata_power: prepared in {prep_elapsed_pow:.0f}s, fill rate {prepared_power['mask'].mean():.2%}")

fdata_skipback_results["fdata_power"] = {}
for arch in FDATA_ARCHS:
    t0 = time.perf_counter()
    result = run_fdata_pair_fixed_lr(prepared_power, "fdata_power", arch,
                                      original_best_lr[("fdata_power", arch)])
    elapsed = time.perf_counter() - t0
    fdata_skipback_results["fdata_power"][arch] = result
    write_skipback_csv("fdata_power", arch, result, prepared_power)
    print(f"fdata_power {arch}: log_R2={result['log_metrics']['R2']:.4f} "
          f"raw_R2={result['raw_metrics']['R2']:.4f} ({elapsed:.1f}s)")

del prepared_power
gc.collect()

dl_pipeline_module.N_TRIALS = 10
print(f"\ndl_pipeline_module.N_TRIALS restored to: {dl_pipeline_module.N_TRIALS}")
print("fdata_power released from memory -- all 6 pairs complete")

fdata_power: 16,315,866 train / 6,992,514 test rows; scaler log-transformed 10/93 columns; sequences built in 462s, fill rate 99.87%
fdata_power: prepared in 806s, fill rate 99.87%


fdata_power LSTM: log_R2=0.9902 raw_R2=0.8708 (5405.2s)


fdata_power TCN: log_R2=0.9912 raw_R2=0.9260 (9487.8s)



dl_pipeline_module.N_TRIALS restored to: 10
fdata_power released from memory -- all 6 pairs complete


## Full comparison: original (`exclude`) vs `skip_back` (fixed lr), original FNN, same-features control

Loaded from real artifacts: original LSTM/TCN/FNN from `data/interim/notebook06_results/`
(written by `06c`/`06d`/`06e`), the same-features control from
`results/notebook06_same_features_control_{combo}_LightGBM.csv` (written by
`06f_dl_comparison.ipynb`), and the `skip_back` fixed-lr results trained live above.

In [9]:
def load_control(combo, model="LightGBM"):
    path = pathlib.Path(f"../results/notebook06_same_features_control_{combo}_{model}.csv")
    control_df = pd.read_csv(path)
    out = {"log_metrics": {}, "raw_metrics": {}}
    for _, row in control_df.iterrows():
        key = "log_metrics" if row["space"] == "log" else "raw_metrics"
        out[key][row["metric"]] = row["value"]
    return out


def metric_row(combo, source, r):
    return dict(combo=combo, source=source, log_R2=r["log_metrics"]["R2"], raw_R2=r["raw_metrics"]["R2"],
                raw_MedAE=r["raw_metrics"]["MedAE"], raw_Within20pct=r["raw_metrics"]["Within20pct"])


comparison_rows = []
for combo in FDATA_COMBOS_IN_ORDER:
    original_fnn = load_pair_artifacts(combo, "FNN")
    control = load_control(combo, "LightGBM")
    comparison_rows.append(metric_row(combo, "original_FNN", original_fnn))
    comparison_rows.append(metric_row(combo, "same_features_control_LightGBM", control))
    for arch in FDATA_ARCHS:
        original = load_pair_artifacts(combo, arch)
        comparison_rows.append(metric_row(combo, f"original_{arch}", original))
        comparison_rows.append(metric_row(combo, f"skipback_{arch}_fixedlr", fdata_skipback_results[combo][arch]))

comparison_df_fdata = pd.DataFrame(comparison_rows).set_index(["combo", "source"])
display(comparison_df_fdata.round(4))

log_R2       raw_R2     raw_MedAE  raw_Within20pct
combo          source                                                                            
fdata_duration original_FNN                    0.8713       0.7661  4.168176e+02          32.4278
               same_features_control_LightGBM  0.8878       0.8360  2.628498e+02          51.9251
               original_LSTM                   0.8291      -3.5850  4.748135e+02          30.0279
               skipback_LSTM_fixedlr           0.8393    -281.0312  4.214391e+02          30.2535
               original_TCN                    0.8345      -3.1166  4.476608e+02          30.3256
               skipback_TCN_fixedlr            0.8599    -199.0371  4.126198e+02          33.4965
fdata_memory   original_FNN                    0.8451       0.8000  6.179679e+08          59.8842
               same_features_control_LightGBM  0.8663       0.8291  4.388359e+08          66.7423
               original_LSTM                   0.7820       0.3846  8.689838e+08          43.9403
               skipback_LSTM_fixedlr           0.8286   -2914.0148  8.800270e+08          50.1504
               original_TCN                    0.7802       0.3965  7.958400e+08          45.0092
               skipback_TCN_fixedlr            0.8216 -210424.6670  6.575191e+08          48.7912
fdata_power    original_FNN                    0.9866       0.7736  1.010300e+01          79.8306
               same_features_control_LightGBM  0.9931       0.9785  5.976500e+00          90.1915
               original_LSTM                   0.9839       0.9382  1.524930e+01          73.4614
               skipback_LSTM_fixedlr           0.9902       0.8708  1.108680e+01          84.2410
               original_TCN                    0.9859       0.9131  1.367310e+01          76.3490
               skipback_TCN_fixedlr            0.9912       0.9260  9.445600e+00          87.4100

## Summary: does `skip_back` (fixed lr) change the F-DATA DL-vs-classical conclusion?

No for `fdata_power`, and if anything more decisively no for `fdata_duration` and
`fdata_memory` -- but the effect is not uniform across the three targets, and the finding
differs in kind between them.

**`fdata_power`: a clean, modest improvement, no collapse.** Log R2 (0.984->0.990 LSTM,
0.986->0.991 TCN), log MedAE, and Within20pct (73.46%->84.24% LSTM, 76.35%->87.41% TCN) all
move in the same favorable direction `06g`'s PM100 power result did. Raw R2 stays strong under
`skip_back`: TCN improves (0.913->0.926), LSTM dips slightly (0.938->0.871) but remains a
good fit -- no catastrophe. A single, previously unflagged user, `usr_2493` (59 rows), holds
63.75%/42.37% of total squared error; excluding it raises raw R2 further still (0.944/0.949),
but this is a minor concentration effect on an already-solid fit, not a new problem. The
original three-user collapse was never a power-specific issue, and it stays negligible here
too (under 0.001% combined share).

**`fdata_duration` and `fdata_memory`: the original three-user collapse (`usr_1794`,
`usr_1848`, `usr_2853` -- 80.7%/82.4% of squared error on duration, 41.5%/44.1% on memory under
the original `exclude` policy) is genuinely resolved under `skip_back`** (combined share drops
to under 0.3% on both targets, for both architectures). Log-space metrics and Within20pct
improve modestly on both, the same direction as PM100 and power. But raw R2 -- already negative
on duration, solidly positive on memory under the original policy -- collapses to catastrophic
levels under `skip_back` (duration: -3.6/-3.1 to -281/-199; memory: +0.38/+0.40 to
-2,914/-210,425). This is driven by two specific, previously unflagged users (`usr_3508`,
5,865 rows; `usr_3611`, 227 rows) that together hold essentially all of the squared error
(97-99%) on both targets and both architectures. Excluding just these two still leaves raw R2
deeply negative on duration and badly negative on memory; the worst 5 users have to be removed
before raw R2 looks sane again.

**Bottom line for the thesis's adopted-model comparison: unchanged.** The already-adopted
classical-ML models (XGBoost/LightGBM, per `results/results_ledger.csv`) remain the right
choice for every F-DATA target under either tie policy -- `skip_back` does not overturn this on
power (DL was already behind classical ML there) and makes the raw-space gap far worse on
duration and memory, where DL's raw-space fit was already the weak point.

**A disclosed limitation that bears directly on how to read this**: every number here comes
from ONE fixed-lr run per pair, with `lr` reused from the original `exclude`-policy study and
never re-searched for `skip_back`'s different sequence construction -- unlike `06g`'s PM100
multi-seed section, which used `skip_back`'s own tuned lr (from the `_skipback`-suffixed Optuna
study) and 3 seeds per pair. Whether the duration/memory raw-R2 collapse is a real property of
`skip_back`'s tie-handling on these two users' specific job histories, or an artifact of running
them through a learning rate tuned for a different sequence construction, is not resolved here.
The fact that the SAME two users (`usr_3508`, `usr_3611`) drive the collapse identically across
both architectures and both targets, while power's analogous concentration (a different single
user, a much smaller effect, no collapse) stays architecture-consistent too, is suggestive but
not conclusive either way -- a genuine open question this notebook does not attempt to resolve,
left for any future investigation that re-tunes `lr` for `skip_back` specifically on these two
users' sequences.

**This is not a controlled comparison, and that matters for every number above.** The
original `exclude`-policy results are each the best of 10 Optuna trials per pair, selected by
log-space validation loss. Every `skip_back` result in this notebook is a single, unselected
run at one fixed `lr` value (reused from the original study, never searched for `skip_back`
itself) -- one draw, not a winner picked from several. Any of `skip_back`'s apparent
differences from `exclude`, in either direction, could simply be what that one reused `lr`
happens to do on `skip_back`'s sequences, not a property of the tie policy itself. `06g`'s
PM100 multi-seed section is the only part of this pair of notebooks that avoids this
asymmetry, since it tunes `lr` separately for `skip_back` and checks 3 seeds per pair.

## Diagnostic: runaway predicted-value check and train-range clipping

**Diagnostic only -- does not replace the headline numbers reported above.** This checks
whether `fdata_duration`/`fdata_memory`'s raw R2 (both the original `exclude` policy's own
already-negative duration raw R2, and `skip_back`'s far worse collapse on both targets) is
being driven by predictions that land far outside the model's own training range in log
space, using ONLY the already-saved test-prediction `.npz` artifacts from `06c`/`06d`/`06e`
(`exclude`) and this notebook's own run above (`skip_back`) -- no retraining, no new model
runs, no new cells executed beyond this diagnostic section.

The train-range bound used for clipping is each combination's own `tune_train` log-target
min/max -- the SAME 13,052,693-row chronological 80% split `run_pair` actually trained on
(not the full 16,315,867-row train split, which also includes the held-out `tune_val` 20%).
Reconstructed directly from each `*_prepared.joblib`'s `adt_epoch` column via the identical
`train_frac=0.8` quantile boundary `src.splits.chronological_split` uses -- confirmed to
reproduce the exact row count (13,052,693) every pair's own metrics CSV already reports as
`fit_rows`, so this is the real split boundary, not an approximation of it. Using train-only
information to pick the clip bound introduces no leakage from the test split.

In [10]:
import pathlib
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import median_absolute_error, r2_score

DIAG_INTERIM = pathlib.Path("../data/interim")
DIAG_THREE_USERS = ["usr_1794", "usr_1848", "usr_2853"]
DIAG_NEW_USERS = ["usr_3508", "usr_3611"]

DIAG_PREPARED_FILE = {
    "fdata_duration": "fdata_duration_r2_prepared.joblib",
    "fdata_memory": "fdata_memory_prepared.joblib",
}
DIAG_POLICY_DIR = {
    "exclude": DIAG_INTERIM / "notebook06_results",
    "skip_back": DIAG_INTERIM / "notebook06h_skipback_results",
}


def diag_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r2 = r2_score(y_true, y_pred)
    nonzero = y_true != 0
    rel_err = np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])
    medae = float(median_absolute_error(y_true, y_pred))
    within20 = float(np.mean(rel_err <= 0.20) * 100)
    return r2, medae, within20


def diag_train_log_range(combo):
    """tune_train log-target min/max, reconstructed from the prepared joblib's own
    adt_epoch column via the same train_frac=0.8 quantile boundary
    src.splits.chronological_split uses -- verified to reproduce fit_rows=13,052,693
    exactly for both fdata_duration and fdata_memory before this cell was added."""
    prep = joblib.load(DIAG_INTERIM / DIAG_PREPARED_FILE[combo])
    X_train = prep["X_train"]
    y_train = np.asarray(prep["y_train"], dtype=np.float64)
    times = X_train["adt_epoch"].to_numpy(dtype=np.float64)
    boundary = np.quantile(times, 0.8)
    tt_mask = times <= boundary
    y_tt = y_train[tt_mask]
    return float(y_tt.min()), float(y_tt.max()), int(tt_mask.sum())


def diag_se_share(user_ids, se, users):
    total = se.sum()
    return {u: 100 * se[user_ids == u].sum() / total for u in users}


diag_rows = []
diag_detail_rows = []
for combo in ["fdata_duration", "fdata_memory"]:
    train_min, train_max, n_tt = diag_train_log_range(combo)
    for arch in ["LSTM", "TCN"]:
        for policy in ["exclude", "skip_back"]:
            fname_arch = arch if policy == "exclude" else f"{arch}_skipback"
            npz_path = DIAG_POLICY_DIR[policy] / f"{combo}_{fname_arch}_test_predictions.npz"
            npz = np.load(npz_path, allow_pickle=True)
            user_ids = npz["user_ids"]
            y_true_raw = npz["y_test_raw"].astype(float)
            preds_log = npz["preds_log"].astype(float)

            max_pred = float(preds_log.max())
            p999 = float(np.quantile(preds_log, 0.999))
            n_exceed = int((preds_log > train_max + 1.0).sum())

            preds_log_clipped = np.clip(preds_log, train_min, train_max)
            preds_raw_clipped = np.expm1(preds_log_clipped)
            r2_c, medae_c, within20_c = diag_regression_metrics(y_true_raw, preds_raw_clipped)

            se_clipped = (y_true_raw - preds_raw_clipped) ** 2
            three_share = sum(diag_se_share(user_ids, se_clipped, DIAG_THREE_USERS).values())
            new_share = sum(diag_se_share(user_ids, se_clipped, DIAG_NEW_USERS).values())

            diag_rows.append(dict(
                combo=combo, arch=arch, policy=policy,
                train_log_min=round(train_min, 4), train_log_max=round(train_max, 4),
                max_pred_log=round(max_pred, 4), p999_pred_log=round(p999, 4),
                n_exceed_trainmax_plus1=n_exceed,
                clipped_raw_R2=round(r2_c, 4), clipped_raw_MedAE=round(medae_c, 2),
                clipped_raw_Within20pct=round(within20_c, 2),
                clipped_three_user_se_share_pct=round(three_share, 4),
                clipped_new_user_se_share_pct=round(new_share, 4),
            ))

diag_df = pd.DataFrame(diag_rows).set_index(["combo", "arch", "policy"])
pd.set_option("display.width", 160)
display(diag_df)

train_log_min  train_log_max  max_pred_log  p999_pred_log  n_exceed_trainmax_plus1  clipped_raw_R2  clipped_raw_MedAE  \
combo          arch policy                                                                                                                             
fdata_duration LSTM exclude           0.6931        12.4654       14.3766        13.5355                     8895         -0.3458       4.748000e+02   
                    skip_back         0.6931        12.4654       18.9398        12.8523                     2172          0.5657       4.214300e+02   
               TCN  exclude           0.6931        12.4654       14.3454        13.4631                     6890         -0.2912       4.476500e+02   
                    skip_back         0.6931        12.4654       18.7300        13.1900                     5000          0.4661       4.126100e+02   
fdata_memory   LSTM exclude          16.6998        25.7185       26.6640        24.8442                        0          0.3850       8.689786e+08   
                    skip_back        16.6998        25.7185       32.7745        24.3974                      332          0.6622       8.800270e+08   
               TCN  exclude          16.6998        25.7185       26.0725        24.8123                        0          0.3966       7.958386e+08   
                    skip_back        16.6998        25.7185       34.9064        24.3069                      333          0.6374       6.575190e+08   

                               clipped_raw_Within20pct  clipped_three_user_se_share_pct  clipped_new_user_se_share_pct  
combo          arch policy                                                                                              
fdata_duration LSTM exclude                      30.08                          69.9994                         0.2393  
                    skip_back                    30.28                          26.5275                         0.4494  
               TCN  exclude                      30.38                          71.7211                         0.2188  
                    skip_back                    33.52                          39.1674                         0.3078  
fdata_memory   LSTM exclude                      43.94                          41.5590                         0.0623  
                    skip_back                    50.15                           3.9348                         3.6406  
               TCN  exclude                      45.01                          44.0787                         0.0783  
                    skip_back                    48.79                           4.0178                         3.3740

**What the diagnostic shows, stated plainly:**

**The `skip_back` collapse is almost entirely a runaway-extrapolation artifact, not a stable
property of the normal-range predictions.** Under `skip_back`, predicted log values reach
18.7-18.9 (duration) and 32.8-34.9 (memory) against a `tune_train` range that tops out at
12.47 (duration) / 25.72 (memory) -- 6 to 9 log-units past anything the model trained on, on
a few hundred to a few thousand rows. Clipping those predictions back to the train range
repairs most of the damage: duration raw R2 goes from -281.03/-199.04 to +0.57/+0.47;
memory's goes from -2,914.01/-210,424.67 to +0.66/+0.64. Once clipped, `skip_back`'s raw R2 is
better than the original `exclude` policy's own clipped raw R2 on both targets (duration:
-0.35/-0.29 clipped-exclude vs +0.57/+0.47 clipped-skip_back; memory: clipping barely moves
`exclude`'s raw R2, even though a few hundred rows do exceed `train_max` itself (just not
by the +1.0 margin counted above): 0.385->0.385 (LSTM), 0.396->0.397 (TCN) -- still far
below `skip_back`'s clipped 0.66/0.64).

**The original three-user problem (`usr_1794`/`usr_1848`/`usr_2853`) re-emerges partially
once the runaway values are capped, but stays well below `exclude`'s level.** Before clipping,
these three users were a negligible 0.14%/0.28% (duration) and 0.0005%/0.0000% (memory) of
squared error under `skip_back` -- not because their own predictions got better, but because
`usr_3508`/`usr_3611`'s runaway predictions were so large they swamped every other row's
contribution to the total. After clipping, the three named users' share rises to 26.5%/39.2%
(duration) and 3.9%/4.0% (memory) -- real, but still far below `exclude`'s clipped share of
70.0%/71.7% (duration) and 41.6%/44.1% (memory, essentially unchanged from `exclude`'s
unclipped share since `exclude` has no rows to clip there). `usr_3508`/`usr_3611`'s own share
drops to under 0.5% (duration) and 3.0-3.7% (memory) once their runaway predictions are capped,
confirming it genuinely was the UNCLIPPED magnitude of their predictions driving the
unclipped headline collapse, not a change in which rows are hardest to predict.

**The original `exclude` policy is not free of this either, just far milder.** On duration,
`exclude` already has 6,890-8,895 test rows (out of 6,992,514) predicting more than one
log-unit past the `tune_train` maximum, which is why its own raw R2 is negative even before
`skip_back` is involved; clipping those rows alone nearly halves the gap to zero (-3.585/-3.117
to -0.346/-0.291). On memory, `exclude` has zero rows exceeding that threshold, so its raw R2
is unaffected by this diagnostic either way.

**This does not overturn the headline comparison, and does not establish `skip_back` as
better.** The headline numbers above are the real, single fixed-lr run on the full, unclipped
test set -- this diagnostic only shows that a large share of `skip_back`'s reported
catastrophic collapse traces to a small number of extreme out-of-range predictions rather than
a uniformly worse fit, and that `exclude` is not entirely clean of the same failure mode on
duration. Whether either policy's extrapolation behavior would change under a learning rate
actually tuned for it is a separate, unresolved question (see the learning-rate caveat in the
summary above).